# A/B Test 통계 검정 실습

이 노트북은 `A와 B는 실제로 다르다`는 상황을 직접 시뮬레이션하고,
`귀무가설: A와 B는 같다`를 어떻게 검정하는지 실습하는 용도다.

이번 실습에서는:

- A의 실제 전환율은 10%
- B의 실제 전환율은 12%

라고 가정한다.

즉, 현실에서는 B가 더 좋지만,
샘플 데이터만 보고도 그 차이를 통계적으로 잡아낼 수 있는지 확인해본다.


In [ ]:
# 수학 함수와 오차 함수(erf)를 쓰기 위한 표준 라이브러리
import math
# 난수 생성과 수치 계산을 위한 라이브러리
import numpy as np
# 표 형태 결과 정리를 위한 라이브러리
import pandas as pd
# 그래프를 그리기 위한 라이브러리
import matplotlib.pyplot as plt

# 그래프를 보기 좋은 스타일로 설정
plt.style.use("ggplot")


## 1. 실험 설정

A와 B의 실제 전환율을 미리 정해두고, 각 그룹에 유저를 배정해 전환 여부를 생성한다.


In [9]:
# 재현 가능한 난수를 만들기 위해 시드를 고정한 난수 생성기 생성
rng = np.random.default_rng(42)

# A 그룹 사용자 수
n_a = 5000
# B 그룹 사용자 수
n_b = 5000

# 실제 세계의 전환율이라고 가정
# A는 10%, B는 12% 확률로 전환된다고 설정
true_p_a = 0.10
true_p_b = 0.12

# 각 사용자에 대해 전환(1) 또는 비전환(0)을 생성
# binomial(1, p)는 베르누이 시행을 size만큼 반복하는 것과 같다
a = rng.binomial(1, true_p_a, size=n_a)
b = rng.binomial(1, true_p_b, size=n_b)

# 실험 설정값을 출력해서 지금 어떤 상황을 시뮬레이션하는지 확인
print(f"true_p_a={true_p_a:.3f}, true_p_b={true_p_b:.3f}")
print(f"A sample size={n_a}, B sample size={n_b}")


true_p_a=0.100, true_p_b=0.120
A sample size=5000, B sample size=5000


## 2. 관측된 전환율 계산

우리는 실제 전환율을 안다고 가정하지 않고, 샘플에서 관측된 전환율만 본다.


In [10]:
# 샘플에서 관측된 A 그룹 전환율 계산
obs_p_a = a.mean()
# 샘플에서 관측된 B 그룹 전환율 계산
obs_p_b = b.mean()
# 우리가 관심 있는 값인 B-A 전환율 차이 계산
diff = obs_p_b - obs_p_a

# 결과를 표 형태로 보기 쉽게 정리
summary_df = pd.DataFrame(
    {
        # 그룹 이름
        "group": ["A", "B"],
        # 각 그룹 사용자 수
        "users": [n_a, n_b],
        # 각 그룹 전환 수
        "conversions": [a.sum(), b.sum()],
        # 각 그룹 관측 전환율
        "conversion_rate": [obs_p_a, obs_p_b],
    }
)

# Jupyter에서는 마지막 변수명을 두면 표가 바로 출력된다
summary_df


,group,users,conversions,conversion_rate
0,A,5000,493,0.0986
1,B,5000,590,0.1180


## 3. Proportion z-test 직접 계산

귀무가설은 `A와 B의 전환율은 같다`이다.

즉:

- H0: p_A = p_B
- H1: p_A != p_B

두 비율 차이에 대한 z-score와 p-value를 직접 계산해본다.


In [11]:
# 표준정규분포의 누적분포함수(CDF)를 계산하는 함수
# 주어진 z값보다 왼쪽 면적이 얼마나 되는지 반환한다
def normal_cdf(x):
    # erf를 이용해 표준정규분포 CDF 계산
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

# 두 비율 차이에 대한 z-test를 직접 구현한 함수
def two_proportion_z_test(x_a, n_a, x_b, n_b):
    # A 그룹 관측 전환율
    p_a = x_a / n_a
    # B 그룹 관측 전환율
    p_b = x_b / n_b
    # 귀무가설 아래에서는 두 그룹 비율이 같다고 보므로
    # 둘을 합친 공통 전환율을 먼저 추정한다
    pooled_p = (x_a + x_b) / (n_a + n_b)
    # 두 비율 차이의 표준오차 계산
    se = math.sqrt(pooled_p * (1 - pooled_p) * (1 / n_a + 1 / n_b))
    # 관측된 차이가 표준오차 대비 몇 배나 큰지 z-score로 계산
    z = (p_b - p_a) / se
    # 양측검정 p-value 계산
    p_value = 2 * (1 - normal_cdf(abs(z)))
    # z-score, p-value, 표준오차 반환
    return z, p_value, se

# 샘플 데이터에서 실제 전환 수 계산
x_a = int(a.sum())
x_b = int(b.sum())

# 실제 샘플 데이터에 대해 z-test 수행
z_score, p_value, se = two_proportion_z_test(x_a, n_a, x_b, n_b)

# 해석에 필요한 관측값 출력
print(f"observed p_a = {obs_p_a:.4f}")
print(f"observed p_b = {obs_p_b:.4f}")
print(f"observed diff (B-A) = {diff:.4f}")
print(f"standard error = {se:.6f}")
print(f"z-score = {z_score:.4f}")
print(f"p-value = {p_value:.6f}")


observed p_a = 0.0986
observed p_b = 0.1180
observed diff (B-A) = 0.0194
standard error = 0.006215
z-score = 3.1214
p-value = 0.001800


## 4. 95% 신뢰구간과 함께 보기

비율 차이 `B-A`에 대한 95% 신뢰구간도 같이 계산해본다.
신뢰구간에 0이 포함되지 않으면, `차이가 없다`는 가설과 잘 맞지 않는다고 볼 수 있다.


In [12]:
# 비율 차이의 신뢰구간은 비결합 표준오차를 많이 사용
# 관측된 A와 B 전환율을 기준으로 차이의 표준오차 계산
se_diff = math.sqrt(obs_p_a * (1 - obs_p_a) / n_a + obs_p_b * (1 - obs_p_b) / n_b)
# 95% 신뢰구간 하한
ci_low = diff - 1.96 * se_diff
# 95% 신뢰구간 상한
ci_high = diff + 1.96 * se_diff

# 전환율 차이(B-A)에 대한 95% 신뢰구간 출력
print(f"95% CI for (B-A): [{ci_low:.4f}, {ci_high:.4f}]")

# 귀무가설에서는 차이 0을 기대하므로
# 0이 신뢰구간 안에 들어가는지 여부를 확인
if ci_low <= 0 <= ci_high:
    print("0이 신뢰구간 안에 있으므로, 차이가 없다는 가설과 크게 충돌하지 않는다.")
else:
    print("0이 신뢰구간 밖에 있으므로, 차이가 없다는 가설과 잘 맞지 않는다.")


95% CI for (B-A): [0.0072, 0.0316]
0이 신뢰구간 밖에 있으므로, 차이가 없다는 가설과 잘 맞지 않는다.


## 5. 결과 해석

이번 샘플에서는 실제로 B가 더 좋게 설정되어 있다.
그런데 샘플만 봐도 그 차이를 잘 잡아내는지 확인한다.


In [13]:
# 보통 5% 유의수준을 많이 사용
alpha = 0.05

# p-value가 유의수준보다 작으면 귀무가설 기각
if p_value < alpha:
    print("귀무가설 기각: A와 B가 같다고 보기 어렵다.")
else:
    print("귀무가설 유지: 이번 샘플만으로는 A와 B가 다르다고 보기 어렵다.")

# 차이의 방향도 함께 확인
if diff > 0:
    print("관측상으로는 B가 A보다 좋아 보인다.")
else:
    print("관측상으로는 B가 A보다 좋아 보이지 않는다.")


귀무가설 기각: A와 B가 같다고 보기 어렵다.
관측상으로는 B가 A보다 좋아 보인다.


## 6. 같은 실험을 여러 번 반복하면 어떻게 될까

한 번의 실험 결과만 보면 우연이 섞일 수 있다.
그래서 같은 실제 전환율 조건에서 실험을 여러 번 반복해 보면,
`관측된 B-A 차이`가 어떤 분포를 가지는지 볼 수 있다.


In [ ]:
# 동일한 실험을 몇 번 반복할지 설정
n_sim = 3000
# 각 반복 실험의 관측 차이(B-A)를 저장할 리스트
diffs = []
# 각 반복 실험의 p-value를 저장할 리스트
p_values = []

# 같은 실제 전환율 조건에서 실험을 여러 번 반복
for _ in range(n_sim):
    # 반복마다 새로운 A 샘플 생성
    a_sim = rng.binomial(1, true_p_a, size=n_a)
    # 반복마다 새로운 B 샘플 생성
    b_sim = rng.binomial(1, true_p_b, size=n_b)

    # 전환 수 계산
    x_a_sim = int(a_sim.sum())
    x_b_sim = int(b_sim.sum())
    # 관측 전환율 계산
    p_a_sim = a_sim.mean()
    p_b_sim = b_sim.mean()

    # 각 반복 실험에 대해 z-test 수행
    z_sim, p_sim, _ = two_proportion_z_test(x_a_sim, n_a, x_b_sim, n_b)
    # 관측 차이 저장
    diffs.append(p_b_sim - p_a_sim)
    # p-value 저장
    p_values.append(p_sim)

# 이후 계산과 시각화를 위해 numpy 배열로 변환
diffs = np.array(diffs)
p_values = np.array(p_values)

# 반복 실험 결과 요약 출력
print(f"평균 관측 차이(B-A): {diffs.mean():.4f}")
print(f"실제 차이(true_p_b - true_p_a): {true_p_b - true_p_a:.4f}")
print(f"유의확률 0.05 기준에서 귀무가설 기각 비율: {(p_values < 0.05).mean():.3f}")


In [ ]:
# 1행 2열 그래프 영역 생성
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽 그래프: 반복 실험에서 나온 B-A 차이의 분포
axes[0].hist(diffs, bins=40, color="#4C78A8", edgecolor="white")
# 차이가 없다는 기준선(0)
axes[0].axvline(0, color="black", linestyle="--", label="no difference")
# 우리가 설정한 실제 차이(0.12 - 0.10 = 0.02)
axes[0].axvline(true_p_b - true_p_a, color="red", linestyle="-", label="true difference")
# 그래프 제목
axes[0].set_title("Distribution of observed (B-A)")
# x축 라벨
axes[0].set_xlabel("Observed difference in conversion rate")
# y축 라벨
axes[0].set_ylabel("Count")
# 범례 표시
axes[0].legend()

# 오른쪽 그래프: 반복 실험에서 나온 p-value 분포
axes[1].hist(p_values, bins=40, color="#F58518", edgecolor="white")
# 보통 쓰는 유의수준 0.05 기준선
axes[1].axvline(0.05, color="black", linestyle="--", label="alpha=0.05")
# 그래프 제목
axes[1].set_title("Distribution of p-values")
# x축 라벨
axes[1].set_xlabel("p-value")
# y축 라벨
axes[1].set_ylabel("Count")
# 범례 표시
axes[1].legend()

# 그래프 간격 정리
plt.tight_layout()
# 그래프 출력
plt.show()


## 7. 직접 바꿔보면 좋은 값

아래 값을 바꿔 보면서 실험해보면 감이 빨리 잡힌다.

- `true_p_a = 0.10`, `true_p_b = 0.11`
- `true_p_a = 0.10`, `true_p_b = 0.12`
- `n_a = n_b = 500`
- `n_a = n_b = 5000`
- `n_a = n_b = 50000`

즉:

- 실제 차이가 작을수록 검정이 어려워지고
- 샘플 수가 많을수록 차이를 더 잘 잡아낸다

는 점을 직접 확인할 수 있다.
